# 0. Импорт библиотек и установка зависимостей

In [3]:
import re
import pandas as pd
from glob import glob
import aiohttp
import aiomoex
from tqdm.autonotebook import tqdm
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger,
    NewsSyntaxParser, NewsNERTagger, Doc
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from matplotlib import pyplot as plt
from nltk.corpus import stopwords
import nltk
from transformers import pipeline
import torch

nltk.download("stopwords")


tqdm.pandas()

/tmp/ipykernel_1055/3622524089.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
[nltk_data] Downloading package stopwords to /home/v0es/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 1. Аггрегация новостей

In [4]:
news_dfs = [pd.read_csv(f) for f in glob('./data/*.csv')]
df = pd.concat(news_dfs, ignore_index=True)

# Приведение дат к нужному формату и сортировка
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values(by='date').reset_index(drop=True)

# Тестовая выборка
df_test = pd.concat([df.head(200), df.tail(200)])

## 1.1 Слияние разделов `finance` и `financies`

In [18]:
df.topic = df.topic.apply(lambda x: 'finance' if x == 'financies' else x)

# 2. Выделение тикеров из текста новости

## 2.1 Инициализация моделей Natasha

In [ ]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)


## 2.2 Необходимые словари

In [ ]:
# Словарь ключевых слов для тикеров голубых фишек
TICKER_KEYWORDS = {
    "SBER": ["сбербанк", "сбер", "пао сбербанк"],
    "GAZP": ["газпром", "пао газпром", "газовая компания"],
    "LKOH": ["лукойл", "пао лукойл", "нк лукойл"],
    "ROSN": ["роснефть", "пао роснефть", "нк роснефть"],
    "YDEX": ["яндекс", "yandex", "яндекс.такси", "яндекс.недвижимость"],
    "VTBR": ["втб", "банк втб", "пао втб"],
    "TATN": ["татнефть", "пао татнефть", "татнефть-нк"],
    "GMKN": ["норильский никель", "пао гмк норильский никель", "gmk norilsk nickel"]
}

## 2.3 Функции обработки текста

In [ ]:
def normalize_text(text: str) -> str:
    """Лемматизация и очистка текста от чисел и символов."""
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    if not doc.tokens:
        return ''
    lemmas = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        if re.match(r'^[а-яА-Яa-zA-Z]+$', token.text):  # исключаем числа и знаки
            lemmas.append(token.lemma.lower())

    return " ".join(lemmas)


def extract_organizations(text: str) -> list[str]:
    """Извлекает организации из текста с помощью Natasha NER."""
    doc = Doc(text)
    doc.segment(segmenter)
    doc.parse_syntax(syntax_parser)
    doc.tag_morph(morph_tagger)
    doc.tag_ner(ner_tagger)
    found_spans = []
    if not doc.spans:
        return []
    for span in doc.spans:
        if span.type == 'ORG':
            span.normalize(morph_vocab)
            found_spans.append(span.normal.lower())
    return found_spans


def extract_tickers(orgs: list[str]) -> list[str]:
    """Определяет тикеры по извлечённым названиям компаний."""
    tickers = set()
    for org in orgs:
        for ticker, keywords in TICKER_KEYWORDS.items():
            if any(org == kw.lower() for kw in keywords):
                tickers.add(ticker)
    return list(tickers)

# 3. Определение сектора экономики, на который влияет новость

У каждого тикера есть список индексов (иными словами - списков, категорий), в которые он входит. Среди этих индексов есть такие, которые показывают сектор экономики, к которому относится компания.

Выберем следующие индексы с мосбиржи:
| Индекс   | Сектор                 |
| -------- | ---------------------- |
| MOEXOG | Нефть и газ            |
| MOEXFN  | Финансовый сектор      |
| MOEXMM  | Металлургия и добыча   |
| MOEXCN  | Потребительский сектор |
| MOEXTN  | Транспорт              |
| MOEXEU  | Электроэнергетика      |
| MOEXTL  | Телекоммуникации      |
| MOEXCH  | Химия и нефтехимия      |
| MOEXIT  | Информационные технологии   |
| MOEXRE  | Строительный сектор   |

Будем определять сектор влияния новости через косинусное расстояние между `tfidf` матрицей из текста новости и матрицей из ключевых слов по секторам
Если нет явного сходства с конкретным сектором, то запишем `ALL`

## 3.1 Настройка TF-IDF для определения сектора

In [ ]:

# Ключевые слова по индексам
INDEX_KEYWORDS = {
    "MOEXOG": [  # Нефть и газ
        "нефть", "газ", "добыча", "бурение", "скважина", "топливо", "трубопровод",
        "нефтепереработка", "нефтехимия", "сланец", "бензин", "дизель", "нефтяная компания"
    ],
    "MOEXFN": [  # Финансовый сектор
        "банк", "кредит", "ставка", "ипотека", "вклад", "депозит", "облигация", "акция",
        "финансы", "страхование", "инвестиции", "ЦБ", "биржа", "дивиденды", "брокер"
    ],
    "MOEXMM": [  # Металлургия и добыча
        "металл", "сталь", "рудник", "шахта", "добыча", "уголь", "железная руда",
        "никель", "медь", "алюминий", "золото", "серебро", "горнодобыча", "плавка"
    ],
    "MOEXCN": [  # Потребительский сектор
        "потребитель", "ритейл", "магазин", "торговля", "продукты", "еда", "напитки",
        "одежда", "FMCG", "бытовая техника", "супермаркет", "розничная торговля", "товары"
    ],
    "MOEXTN": [  # Транспорт
        "транспорт", "логистика", "авиакомпания", "железная дорога", "порт", "флот",
        "аэропорт", "автоперевозки", "грузоперевозки", "контейнер", "трамвай", "метро"
    ],
    "MOEXEU": [  # Электроэнергетика
        "электроэнергия", "энергетика", "генерация", "электростанция", "теплоэнергия",
        "ГЭС", "ТЭЦ", "АЭС", "мощность", "энергосбыт", "электричество", "энергокомпания"
    ],
    "MOEXTL": [  # Телекоммуникации
        "телеком", "связь", "интернет", "мобильный оператор", "сотовая связь",
        "телефон", "передача данных", "оптоволокно", "трафик", "сетевые технологии"
    ],
    "MOEXCH": [  # Химия и нефтехимия
        "химия", "нефтехимия", "пластик", "полимер", "удобрения", "реактивы", "синтез",
        "нефтепродукты", "катализатор", "химическое производство", "серная кислота"
    ],
    "MOEXIT": [  # Информационные технологии
        "it", "айти", "технологии", "программное обеспечение", "разработка", "интернет-сервис",
        "стартап", "искусственный интеллект", "кибербезопасность", "софт", "облачные технологии",
        "приложение", "данные", "автоматизация", "цифровизация"
    ],
    "MOEXRE": [  # Строительные компании
        "стройка", "строительство", "девелопмент", "недвижимость", "жилой комплекс",
        "инфраструктура", "ремонт", "архитектура", "подрядчик", "застройщик", "объект", "капитальное строительство"
    ]
}

BASE_INDEX_URL = 'https://iss.moex.com/iss/securities/{ticker}/indices.json?only_actual=1'


russian_stopwords = stopwords.words("russian")

sector_names = list(INDEX_KEYWORDS.keys())
sector_docs = [" ".join(words) for words in INDEX_KEYWORDS.values()]
sector_docs_norm = [normalize_text(doc) for doc in sector_docs]

vectorizer = TfidfVectorizer(max_features = 10000, stop_words=russian_stopwords)
sector_matrix = vectorizer.fit_transform(sector_docs_norm)


## 3.2 Функция для определения сектора экономики

In [ ]:
def detect_sector_tfidf(text: str) -> str:
    """
    Определяет сектор новости по косинусному сходству с TF-IDF.
    Если нет явного сходства - влияет на общий рынок
    """
    text_norm = normalize_text(text)
    text_vector = vectorizer.transform([text_norm])
    sims = cosine_similarity(text_vector, sector_matrix)[0]
    if sims.max() < 0.4:
        return 'ALL'
    return sector_names[sims.argmax()]


# 4. Определение тональности новости

## 4.1 Инициализация модели

In [15]:

model = pipeline(model="mxlcw/rubert-tiny2-russian-economic-sentiment")

Device set to use cuda:0


In [25]:
res = model(df_test.text.to_list())

In [26]:
res

[{'label': 'positive', 'score': 0.9999397993087769},
 {'label': 'neutral', 'score': 0.999993085861206},
 {'label': 'neutral', 'score': 0.9956881403923035},
 {'label': 'negative', 'score': 0.9999608993530273},
 {'label': 'negative', 'score': 0.9999852180480957},
 {'label': 'negative', 'score': 0.999996542930603},
 {'label': 'negative', 'score': 0.9999896287918091},
 {'label': 'negative', 'score': 0.9999648332595825},
 {'label': 'negative', 'score': 0.9992863535881042},
 {'label': 'negative', 'score': 0.9996289014816284},
 {'label': 'negative', 'score': 0.9999831914901733},
 {'label': 'positive', 'score': 0.999804675579071},
 {'label': 'negative', 'score': 0.9999960660934448},
 {'label': 'negative', 'score': 0.9999783039093018},
 {'label': 'positive', 'score': 0.9999980926513672},
 {'label': 'positive', 'score': 0.9999566078186035},
 {'label': 'positive', 'score': 0.9999971389770508},
 {'label': 'positive', 'score': 0.9999978542327881},
 {'label': 'positive', 'score': 0.9895958304405212}